MMD+SCL

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

acoustic_dataset_1 = pd.read_csv('/content/drive/MyDrive/Acoustic-subject-level-dataset.csv')
acoustic_dataset = acoustic_dataset_1.drop(columns=["Participant_ID"])
print(acoustic_dataset.shape)
print(acoustic_dataset.head())

(193, 70)
   MDVP:Fo(Hz)  MDVP:Fhi(Hz)  MDVP:Flo(Hz)  MDVP:Jitter(%)  MDVP:Jitter(Abs)  \
0   118.719333    141.312833    106.029833        0.010085          0.000087   
1   113.014000    154.201333    105.652667        0.008043          0.000072   
2   223.657833    239.034500    161.921500        0.002585          0.000010   
3   141.336667    165.070500     66.233833        0.009008          0.000065   
4   112.839667    208.860000    107.392833        0.004850          0.000042   

   MDVP:RAP  MDVP:PPQ  Jitter:DDP  MDVP:Shimmer  MDVP:Shimmer(dB)  ...  \
0  0.004998  0.007312    0.014992      0.053932          0.515167  ...   
1  0.003508  0.005035    0.010527      0.029677          0.288667  ...   
2  0.001463  0.001425    0.004393      0.017093          0.151333  ...   
3  0.005090  0.004368    0.015263      0.042640          0.407333  ...   
4  0.002468  0.002817    0.007405      0.025015          0.224500  ...   

   Delta4  Delta5  Delta6  Delta7  Delta8  Delta9  Delta10  Delt

In [3]:
import pandas as pd

meg_dataset = pd.read_csv('/content/drive/MyDrive/final_meg_rest_21_par-lev_final.csv')

print("Loaded data shape:", meg_dataset.shape)
print(meg_dataset.head())

Loaded data shape: (127, 106)
   burst_rate_L_mean  burst_rate_L_std  burst_rate_R_mean  burst_rate_R_std  \
0           4.412365          0.242544           4.486517          0.188241   
1           3.828010          0.316999           3.783657          0.297773   
2           4.645359          0.130894           4.713304          0.111269   
3           4.142208          0.243497           4.183916          0.217601   
4           3.923527          0.336133           4.035406          0.306960   

   burst_rate_asym  mean_burst_duration_L_mean  mean_burst_duration_L_std  \
0        -0.008333                    0.056832                   0.003237   
1         0.005827                    0.065725                   0.005382   
2        -0.007260                    0.053844                   0.001551   
3        -0.005009                    0.060555                   0.003554   
4        -0.014057                    0.064178                   0.005454   

   mean_burst_duration_R_mean  m

MMD+SCL

In [4]:

# ============================================================
# LEAKAGE-FREE RF-SHAP + MMD + SCL + MLP
# CROSS-MODAL + CROSS-PARTICIPANT ANALYSIS
#
# MMD + SCL:
#   1. Maximum Mean Discrepancy aligns acoustic and MEG
#      latent distributions.
#   2. Supervised Contrastive Learning clusters participants
#      according to disease status (HC/PD).
#   3. Task classification loss predicts HC versus PD.
#
# The original preprocessing, RF-SHAP, shared residual
# encoder, 32-D embeddings, MLP, OOF outputs, metrics,
# and CSV filenames are retained.
#
# Acoustic and MEG participants are unpaired.
# No participant matching is required for MMD.
#
# Assumptions:
#   acoustic_dataset: full-feature acoustic DataFrame
#   meg_dataset:      full-feature MEG DataFrame
#   Last column: binary label (0=HC, 1=PD)
# ============================================================

import random
import warnings

import numpy as np
import pandas as pd
import shap

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# DEVICE AND REPRODUCIBILITY
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# MODEL SETTINGS
# ============================================================

LATENT_DIM = 32
TOP_K = 7
N_SPLITS = 5

# RF-SHAP
RF_N_ESTIMATORS = 300

# Numerical safety
NUMERICAL_CAP = 1e12

# MMD + SCL
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
ENCODER_EPOCHS = 120

TASK_WEIGHT = 3.0

# Replaces DOMAIN_WEIGHT from the DANN model.
MMD_WEIGHT = 0.25

SCL_WEIGHT = 0.55
SCL_TEMPERATURE = 0.07

# MMD RBF kernel bandwidth multipliers.
MMD_KERNEL_SCALES = (0.5, 1.0, 2.0, 4.0, 8.0)

# Final MLP
MLP_EPOCHS = 80
MLP_LEARNING_RATE = 1e-3


# ============================================================
# LOAD FULL-FEATURE DATASETS
# ============================================================

df_acoustic_full = acoustic_dataset.copy()
df_meg_full = meg_dataset.copy()


def extract_features_and_labels(df, dataset_name):

    if df.shape[1] < 2:
        raise ValueError(
            f"{dataset_name} must contain features and a label."
        )

    y = pd.to_numeric(
        df.iloc[:, -1],
        errors="raise"
    ).to_numpy(dtype=np.int64)

    X_df = df.iloc[:, :-1].copy()

    X_df = X_df.apply(
        pd.to_numeric,
        errors="coerce"
    )

    X = X_df.to_numpy(dtype=np.float64)

    X[~np.isfinite(X)] = np.nan

    if not np.isin(y, [0, 1]).all():
        raise ValueError(
            f"{dataset_name}: labels must be 0 and 1."
        )

    if X.shape[1] == 0:
        raise ValueError(
            f"{dataset_name}: no candidate features found."
        )

    if len(y) < N_SPLITS:
        raise ValueError(
            f"{dataset_name}: fewer samples than CV folds."
        )

    if np.min(np.bincount(y, minlength=2)) < N_SPLITS:
        raise ValueError(
            f"{dataset_name}: each class must have at least "
            f"{N_SPLITS} participants for stratified CV."
        )

    return X, y, list(X_df.columns)


X_acoustic_raw, y_acoustic, acoustic_feature_names = (
    extract_features_and_labels(
        df_acoustic_full,
        "Acoustic"
    )
)

X_meg_raw, y_meg, meg_feature_names = (
    extract_features_and_labels(
        df_meg_full,
        "MEG"
    )
)

print("\nDataset shapes")
print("Acoustic:", X_acoustic_raw.shape)
print("MEG     :", X_meg_raw.shape)

print(
    "Acoustic labels:",
    dict(zip(*np.unique(y_acoustic, return_counts=True)))
)

print(
    "MEG labels:",
    dict(zip(*np.unique(y_meg, return_counts=True)))
)


# ============================================================
# LEAKAGE-FREE NUMERICAL PREPROCESSING
# ============================================================

def preprocess_fold_train_test(X_train, X_test):

    X_train = np.asarray(
        X_train,
        dtype=np.float64
    ).copy()

    X_test = np.asarray(
        X_test,
        dtype=np.float64
    ).copy()

    if X_train.ndim != 2 or X_test.ndim != 2:
        raise ValueError(
            "Training and test features must be 2-D arrays."
        )

    if X_train.shape[1] != X_test.shape[1]:
        raise ValueError(
            "Training and test feature counts do not match."
        )

    # 1. Replace non-finite values with NaN.
    X_train[~np.isfinite(X_train)] = np.nan
    X_test[~np.isfinite(X_test)] = np.nan

    # 2. Apply the fixed numerical safety cap.
    X_train = np.clip(
        X_train,
        -NUMERICAL_CAP,
        NUMERICAL_CAP
    )

    X_test = np.clip(
        X_test,
        -NUMERICAL_CAP,
        NUMERICAL_CAP
    )

    n_features = X_train.shape[1]

    lower_bounds = np.zeros(n_features, dtype=np.float64)
    upper_bounds = np.zeros(n_features, dtype=np.float64)

    # 3. Learn clipping bounds from training data only.
    for j in range(n_features):

        finite_values = X_train[
            np.isfinite(X_train[:, j]), j
        ]

        if finite_values.size == 0:
            lower_bounds[j] = 0.0
            upper_bounds[j] = 0.0
        else:
            lower_bounds[j] = np.quantile(
                finite_values, 0.01
            )

            upper_bounds[j] = np.quantile(
                finite_values, 0.99
            )

    # 4. Clip both sets using training-derived bounds.
    X_train = np.maximum(
        np.minimum(X_train, upper_bounds),
        lower_bounds
    )

    X_test = np.maximum(
        np.minimum(X_test, upper_bounds),
        lower_bounds
    )

    # 5. Fit training-fold medians.
    medians = np.zeros(n_features, dtype=np.float64)

    for j in range(n_features):

        valid_values = X_train[
            np.isfinite(X_train[:, j]), j
        ]

        if valid_values.size == 0:
            medians[j] = 0.0
        else:
            medians[j] = np.median(valid_values)

    # 6. Apply the fitted medians.
    def impute_with_training_medians(X, fitted_medians):

        X_out = np.asarray(
            X,
            dtype=np.float64
        ).copy()

        X_out[~np.isfinite(X_out)] = np.nan

        for j in range(X_out.shape[1]):

            missing = ~np.isfinite(X_out[:, j])

            if missing.any():
                X_out[missing, j] = fitted_medians[j]

        return X_out

    X_train_clean = impute_with_training_medians(
        X_train, medians
    )

    X_test_clean = impute_with_training_medians(
        X_test, medians
    )

    # 7. Validate outputs.
    if not np.isfinite(X_train_clean).all():
        raise ValueError(
            "Non-finite values remain in training data."
        )

    if not np.isfinite(X_test_clean).all():
        raise ValueError(
            "Non-finite values remain in test data."
        )

    if (
        np.max(np.abs(X_train_clean)) > NUMERICAL_CAP
        or np.max(np.abs(X_test_clean)) > NUMERICAL_CAP
    ):
        raise ValueError(
            "Numerical safety cap was exceeded."
        )

    return (
        X_train_clean,
        X_test_clean,
        medians,
        lower_bounds,
        upper_bounds
    )


# ============================================================
# FOLD-WISE RF-SHAP FEATURE SELECTION
# ============================================================

def select_top_features_rf_shap(
    X_train,
    y_train,
    feature_names,
    top_k=7,
    random_state=42,
    dataset_name="Dataset"
):

    X_train = np.asarray(X_train, dtype=np.float64)
    y_train = np.asarray(y_train, dtype=np.int64)

    if not np.isfinite(X_train).all():
        raise ValueError(
            f"{dataset_name}: non-finite values before RF fitting."
        )

    if np.max(np.abs(X_train)) > np.finfo(np.float32).max:
        raise ValueError(
            f"{dataset_name}: values exceed float32 range."
        )

    n_features = X_train.shape[1]
    k = min(top_k, n_features)

    if k == 0:
        raise ValueError(
            f"{dataset_name}: no candidate features."
        )

    rf = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        random_state=random_state,
        class_weight="balanced",
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_train)

    # Handle different SHAP output formats.
    if isinstance(shap_values, list):

        class_idx = 1 if len(shap_values) > 1 else 0

        selected_shap_values = np.asarray(
            shap_values[class_idx]
        )

    else:

        selected_shap_values = np.asarray(shap_values)

        if selected_shap_values.ndim == 3:

            # Layout: samples x features x classes.
            if (
                selected_shap_values.shape[0] == len(X_train)
                and selected_shap_values.shape[1] == n_features
            ):

                class_idx = min(
                    1, selected_shap_values.shape[2] - 1
                )

                selected_shap_values = (
                    selected_shap_values[:, :, class_idx]
                )

            # Layout: classes x samples x features.
            elif (
                selected_shap_values.shape[1] == len(X_train)
                and selected_shap_values.shape[2] == n_features
            ):

                class_idx = min(
                    1, selected_shap_values.shape[0] - 1
                )

                selected_shap_values = (
                    selected_shap_values[class_idx, :, :]
                )

            else:
                raise ValueError(
                    f"{dataset_name}: unexpected SHAP shape "
                    f"{selected_shap_values.shape}"
                )

    if selected_shap_values.shape != X_train.shape:
        raise ValueError(
            f"{dataset_name}: SHAP shape "
            f"{selected_shap_values.shape} does not match "
            f"training data {X_train.shape}."
        )

    if not np.isfinite(selected_shap_values).all():
        raise ValueError(
            f"{dataset_name}: SHAP returned non-finite values."
        )

    shap_importance = np.mean(
        np.abs(selected_shap_values),
        axis=0
    )

    selected_indices = np.argsort(
        shap_importance
    )[::-1][:k]

    selected_names = [
        feature_names[i] for i in selected_indices
    ]

    print(f"\n{dataset_name} RF-SHAP Top-{k} features:")

    for rank, idx in enumerate(selected_indices, start=1):
        print(
            f"{rank:02d}. {feature_names[idx]} "
            f"(mean |SHAP| = {shap_importance[idx]:.6f})"
        )

    return selected_indices, selected_names


# ============================================================
# RESIDUAL BLOCK
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, x):
        return F.relu(self.block(x) + x)


# ============================================================
# MMD + SCL MODEL
# ============================================================

class MMDContrastive(nn.Module):

    def __init__(self, input_dim, latent_dim=32):

        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),

            ResidualBlock(128),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(64, latent_dim)
        )

        self.task_head = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(32, 2)
        )

    def forward(self, x):

        z = self.encoder(x)

        z_norm = F.normalize(
            z,
            p=2,
            dim=1
        )

        task_out = self.task_head(z_norm)

        return task_out, z_norm


# ============================================================
# MAXIMUM MEAN DISCREPANCY (MMD) LOSS
# ============================================================
#
# Uses a multi-scale RBF kernel.
#
# source: latent embeddings from one modality
# target: latent embeddings from the other modality
#
# This compares distributions, not participant pairs.
# The biased empirical MMD^2 estimate is used here.
# ============================================================

def mmd_rbf_loss(
    source,
    target,
    kernel_scales=MMD_KERNEL_SCALES
):

    if source.ndim != 2 or target.ndim != 2:
        raise ValueError(
            "MMD inputs must be two-dimensional."
        )

    if source.shape[1] != target.shape[1]:
        raise ValueError(
            "MMD inputs must have the same latent dimension."
        )

    if source.shape[0] == 0 or target.shape[0] == 0:
        # No cross-domain comparison can be computed in this batch.
        return (source.sum() + target.sum()) * 0.0

    # Pairwise squared Euclidean distances.
    xx = torch.cdist(source, source, p=2).pow(2)
    yy = torch.cdist(target, target, p=2).pow(2)
    xy = torch.cdist(source, target, p=2).pow(2)

    # Estimate bandwidth from the current batch distances.
    with torch.no_grad():

        combined_distances = torch.cat([
            xx.detach().reshape(-1),
            yy.detach().reshape(-1),
            xy.detach().reshape(-1)
        ])

        positive_distances = combined_distances[
            combined_distances > 1e-12
        ]

        if positive_distances.numel() > 0:
            base_bandwidth = torch.median(
                positive_distances
            )
        else:
            base_bandwidth = torch.tensor(
                1.0,
                device=source.device,
                dtype=source.dtype
            )

        base_bandwidth = base_bandwidth.clamp_min(1e-6)

    kernel_xx = 0.0
    kernel_yy = 0.0
    kernel_xy = 0.0

    for scale in kernel_scales:

        bandwidth = (
            base_bandwidth * float(scale)
        ).clamp_min(1e-6)

        k_xx = torch.exp(-xx / (2.0 * bandwidth))
        k_yy = torch.exp(-yy / (2.0 * bandwidth))
        k_xy = torch.exp(-xy / (2.0 * bandwidth))

        kernel_xx = kernel_xx + k_xx
        kernel_yy = kernel_yy + k_yy
        kernel_xy = kernel_xy + k_xy

    n_kernels = float(len(kernel_scales))

    kernel_xx = kernel_xx / n_kernels
    kernel_yy = kernel_yy / n_kernels
    kernel_xy = kernel_xy / n_kernels

    mmd_squared = (
        kernel_xx.mean()
        + kernel_yy.mean()
        - 2.0 * kernel_xy.mean()
    )

    # Numerical noise can cause a tiny negative value.
    # Clamp to preserve the nonnegative MMD^2 objective.
    return torch.clamp(mmd_squared, min=0.0)


# ============================================================
# SUPERVISED CONTRASTIVE LOSS
# ============================================================

class SupConLoss(nn.Module):

    def __init__(self, temperature=0.07):

        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):

        labels = labels.contiguous().view(-1, 1)

        mask = torch.eq(
            labels,
            labels.T
        ).float().to(features.device)

        similarity = torch.matmul(
            features,
            features.T
        ) / self.temperature

        logits_max, _ = torch.max(
            similarity,
            dim=1,
            keepdim=True
        )

        logits = similarity - logits_max.detach()

        logits_mask = (
            torch.ones_like(mask)
            -
            torch.eye(
                mask.shape[0],
                device=features.device
            )
        )

        mask = mask * logits_mask

        exp_logits = (
            torch.exp(logits) * logits_mask
        )

        log_prob = logits - torch.log(
            exp_logits.sum(dim=1, keepdim=True) + 1e-8
        )

        positive_count = mask.sum(dim=1)

        mean_log_prob_pos = (
            (mask * log_prob).sum(dim=1)
            /
            (positive_count + 1e-8)
        )

        valid_anchors = positive_count > 0

        if not valid_anchors.any():
            return features.sum() * 0.0

        return -mean_log_prob_pos[valid_anchors].mean()


# ============================================================
# TRAIN MMD + SCL ON OUTER-TRAINING PARTICIPANTS ONLY
# ============================================================

def train_mmd_scl(
    X_acoustic_train,
    y_acoustic_train,
    X_meg_train,
    y_meg_train,
    fold_number
):

    if X_acoustic_train.shape[1] != X_meg_train.shape[1]:
        raise ValueError(
            "Acoustic and MEG must have the same selected "
            "feature count for the shared encoder."
        )

    X_train_all = np.vstack([
        X_acoustic_train,
        X_meg_train
    ]).astype(np.float32)

    y_train_all = np.hstack([
        y_acoustic_train,
        y_meg_train
    ]).astype(np.int64)

    domain_train_all = np.hstack([
        np.zeros(
            len(y_acoustic_train),
            dtype=np.int64
        ),
        np.ones(
            len(y_meg_train),
            dtype=np.int64
        )
    ]).astype(np.int64)

    if not np.isfinite(X_train_all).all():
        raise ValueError(
            f"Non-finite MMD input in fold {fold_number}."
        )

    dataset = TensorDataset(
        torch.tensor(
            X_train_all,
            dtype=torch.float32
        ),
        torch.tensor(
            y_train_all,
            dtype=torch.long
        ),
        torch.tensor(
            domain_train_all,
            dtype=torch.long
        )
    )

    loader_generator = torch.Generator()
    loader_generator.manual_seed(SEED + fold_number)

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        generator=loader_generator
    )

    model = MMDContrastive(
        input_dim=X_train_all.shape[1],
        latent_dim=LATENT_DIM
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    task_loss_fn = nn.CrossEntropyLoss()

    contrastive_loss_fn = SupConLoss(
        temperature=SCL_TEMPERATURE
    )

    print(
        f"\nTraining MMD + SCL | Outer fold {fold_number}"
    )

    for epoch in range(ENCODER_EPOCHS):

        model.train()

        total_correct = 0
        total_samples = 0
        total_loss = 0.0
        total_task_loss = 0.0
        total_mmd_loss = 0.0
        total_scl_loss = 0.0

        for xb, y_task, y_dom in loader:

            xb = xb.to(device)
            y_task = y_task.to(device)
            y_dom = y_dom.to(device)

            optimizer.zero_grad()

            task_out, z = model(xb)

            # 1. Disease classification loss.
            loss_task = task_loss_fn(
                task_out,
                y_task
            )

            # 2. Separate embeddings by modality.
            acoustic_mask = (y_dom == 0)
            meg_mask = (y_dom == 1)

            z_acoustic = z[acoustic_mask]
            z_meg = z[meg_mask]

            # 3. MMD aligns the modality distributions.
            loss_mmd = mmd_rbf_loss(
                z_acoustic,
                z_meg
            )

            # 4. SCL uses disease labels, not modality labels.
            loss_contrastive = contrastive_loss_fn(
                z,
                y_task
            )

            # 5. Combined objective.
            loss = (
                TASK_WEIGHT * loss_task
                +
                MMD_WEIGHT * loss_mmd
                +
                SCL_WEIGHT * loss_contrastive
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss in fold {fold_number}, "
                    f"epoch {epoch + 1}."
                )

            loss.backward()
            optimizer.step()

            preds = task_out.argmax(dim=1)

            total_correct += (
                preds == y_task
            ).sum().item()

            total_samples += len(y_task)

            total_loss += loss.item()
            total_task_loss += loss_task.item()
            total_mmd_loss += loss_mmd.item()
            total_scl_loss += loss_contrastive.item()

        train_acc = (
            total_correct / max(total_samples, 1)
        )

        if epoch == 0 or (epoch + 1) % 10 == 0:

            n_batches = max(len(loader), 1)

            print(
                f"Epoch {epoch + 1:03d}/{ENCODER_EPOCHS} "
                f"| Loss={total_loss / n_batches:.4f} "
                f"| Task={total_task_loss / n_batches:.4f} "
                f"| MMD={total_mmd_loss / n_batches:.4f} "
                f"| SCL={total_scl_loss / n_batches:.4f} "
                f"| TrainAcc={train_acc:.4f}"
            )

    return model


# ============================================================
# EXTRACT LATENT FEATURES
# ============================================================

def extract_latent(model, X):

    model.eval()

    X = np.asarray(
        X,
        dtype=np.float32
    )

    if not np.isfinite(X).all():
        raise ValueError(
            "Non-finite values passed to latent extraction."
        )

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():
        _, z = model(X_tensor)

    z = z.cpu().numpy().astype(np.float32)

    if not np.isfinite(z).all():
        raise RuntimeError(
            "Non-finite latent embeddings were generated."
        )

    return z


# ============================================================
# FINAL MLP CLASSIFIER
# ============================================================

class FinalMLP(nn.Module):

    def __init__(self, input_dim=32):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.net(x)


def train_and_evaluate_mlp(
    X_train,
    y_train,
    X_test,
    y_test,
    fold_number
):

    X_train = np.asarray(
        X_train,
        dtype=np.float32
    )

    X_test = np.asarray(
        X_test,
        dtype=np.float32
    )

    if not (
        np.isfinite(X_train).all()
        and np.isfinite(X_test).all()
    ):
        raise ValueError(
            "Non-finite latent values passed to the MLP."
        )

    X_train_tensor = torch.tensor(
        X_train,
        dtype=torch.float32,
        device=device
    )

    X_test_tensor = torch.tensor(
        X_test,
        dtype=torch.float32,
        device=device
    )

    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.long,
        device=device
    )

    clf = FinalMLP(
        input_dim=LATENT_DIM
    ).to(device)

    optimizer_clf = optim.Adam(
        clf.parameters(),
        lr=MLP_LEARNING_RATE
    )

    criterion = nn.CrossEntropyLoss()

    clf.train()

    for _ in range(MLP_EPOCHS):

        optimizer_clf.zero_grad()

        outputs = clf(X_train_tensor)

        loss = criterion(
            outputs,
            y_train_tensor
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite MLP loss in fold {fold_number}."
            )

        loss.backward()
        optimizer_clf.step()

    # Evaluate held-out participants only.
    clf.eval()

    with torch.no_grad():

        outputs = clf(X_test_tensor)

        probs = F.softmax(
            outputs,
            dim=1
        )[:, 1].cpu().numpy()

        preds = outputs.argmax(
            dim=1
        ).cpu().numpy()

    y_test = np.asarray(
        y_test,
        dtype=np.int64
    )

    acc = accuracy_score(y_test, preds)

    prec = precision_score(
        y_test,
        preds,
        zero_division=0
    )

    rec = recall_score(
        y_test,
        preds,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        preds,
        zero_division=0
    )

    auc = (
        roc_auc_score(y_test, probs)
        if len(np.unique(y_test)) == 2
        else np.nan
    )

    print(f"\nFold {fold_number}")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"AUC       : {auc:.4f}")

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc,
        "predictions": preds,
        "probabilities": probs
    }


# ============================================================
# OUTER CROSS-VALIDATION SPLITS
# ============================================================

skf_acoustic = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

skf_meg = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

acoustic_splits = list(
    skf_acoustic.split(
        X_acoustic_raw,
        y_acoustic
    )
)

meg_splits = list(
    skf_meg.split(
        X_meg_raw,
        y_meg
    )
)


# ============================================================
# INITIALIZE OOF ARRAYS
# ============================================================

oof_pred_acoustic = np.zeros(
    len(y_acoustic),
    dtype=np.int64
)

oof_prob_acoustic = np.zeros(
    len(y_acoustic),
    dtype=np.float32
)

oof_pred_meg = np.zeros(
    len(y_meg),
    dtype=np.int64
)

oof_prob_meg = np.zeros(
    len(y_meg),
    dtype=np.float32
)

oof_count_acoustic = np.zeros(
    len(y_acoustic),
    dtype=np.int64
)

oof_count_meg = np.zeros(
    len(y_meg),
    dtype=np.int64
)

fused_acoustic_32D = np.zeros(
    (len(y_acoustic), LATENT_DIM),
    dtype=np.float32
)

fused_meg_32D = np.zeros(
    (len(y_meg), LATENT_DIM),
    dtype=np.float32
)

fold_metrics = []
selected_features_by_fold = {}

print("\n" + "=" * 75)
print("LEAKAGE-FREE RF-SHAP + MMD + SCL + MLP")
print("=" * 75)


# ============================================================
# OUTER FOLD LOOP
# ============================================================

for fold in range(N_SPLITS):

    fold_number = fold + 1

    print("\n" + "=" * 75)
    print(f"OUTER FOLD {fold_number}/{N_SPLITS}")
    print("=" * 75)

    # 1. Outer train/test splits.
    train_a_idx, test_a_idx = acoustic_splits[fold]
    train_m_idx, test_m_idx = meg_splits[fold]

    X_a_train_raw = X_acoustic_raw[train_a_idx]
    X_a_test_raw = X_acoustic_raw[test_a_idx]

    y_a_train = y_acoustic[train_a_idx]
    y_a_test = y_acoustic[test_a_idx]

    X_m_train_raw = X_meg_raw[train_m_idx]
    X_m_test_raw = X_meg_raw[test_m_idx]

    y_m_train = y_meg[train_m_idx]
    y_m_test = y_meg[test_m_idx]

    print(
        f"Acoustic train/test: "
        f"{len(train_a_idx)}/{len(test_a_idx)}"
    )

    print(
        f"MEG train/test     : "
        f"{len(train_m_idx)}/{len(test_m_idx)}"
    )

    # 2. Acoustic preprocessing.
    (
        X_a_train_clean,
        X_a_test_clean,
        acoustic_medians,
        acoustic_lower_bounds,
        acoustic_upper_bounds
    ) = preprocess_fold_train_test(
        X_a_train_raw,
        X_a_test_raw
    )

    # 3. MEG preprocessing.
    (
        X_m_train_clean,
        X_m_test_clean,
        meg_medians,
        meg_lower_bounds,
        meg_upper_bounds
    ) = preprocess_fold_train_test(
        X_m_train_raw,
        X_m_test_raw
    )

    print(
        "Acoustic finite after preprocessing:",
        np.isfinite(X_a_train_clean).all()
        and np.isfinite(X_a_test_clean).all()
    )

    print(
        "MEG finite after preprocessing:",
        np.isfinite(X_m_train_clean).all()
        and np.isfinite(X_m_test_clean).all()
    )

    # 4. RF-SHAP Top-7: acoustic training fold only.
    a_selected_idx, a_selected_names = (
        select_top_features_rf_shap(
            X_train=X_a_train_clean,
            y_train=y_a_train,
            feature_names=acoustic_feature_names,
            top_k=TOP_K,
            random_state=SEED + fold_number,
            dataset_name=f"Acoustic fold {fold_number}"
        )
    )

    X_a_train_selected = X_a_train_clean[:, a_selected_idx]
    X_a_test_selected = X_a_test_clean[:, a_selected_idx]

    # 5. RF-SHAP Top-7: MEG training fold only.
    m_selected_idx, m_selected_names = (
        select_top_features_rf_shap(
            X_train=X_m_train_clean,
            y_train=y_m_train,
            feature_names=meg_feature_names,
            top_k=TOP_K,
            random_state=SEED + 100 + fold_number,
            dataset_name=f"MEG fold {fold_number}"
        )
    )

    X_m_train_selected = X_m_train_clean[:, m_selected_idx]
    X_m_test_selected = X_m_test_clean[:, m_selected_idx]

    if len(a_selected_idx) != len(m_selected_idx):
        raise ValueError(
            "Acoustic and MEG must have equal selected feature "
            "counts for the shared-input encoder."
        )

    selected_features_by_fold[fold_number] = {
        "acoustic": a_selected_names,
        "meg": m_selected_names
    }

    # 6. Fold-wise standardization.
    scaler_acoustic = StandardScaler()

    X_a_train = scaler_acoustic.fit_transform(
        X_a_train_selected
    )

    X_a_test = scaler_acoustic.transform(
        X_a_test_selected
    )

    scaler_meg = StandardScaler()

    X_m_train = scaler_meg.fit_transform(
        X_m_train_selected
    )

    X_m_test = scaler_meg.transform(
        X_m_test_selected
    )

    for data_name, data_array in [
        ("Acoustic train", X_a_train),
        ("Acoustic test", X_a_test),
        ("MEG train", X_m_train),
        ("MEG test", X_m_test)
    ]:

        if not np.isfinite(data_array).all():
            raise RuntimeError(
                f"{data_name} contains non-finite values "
                f"in fold {fold_number} after standardization."
            )

    X_a_train = X_a_train.astype(np.float32)
    X_a_test = X_a_test.astype(np.float32)

    X_m_train = X_m_train.astype(np.float32)
    X_m_test = X_m_test.astype(np.float32)

    # 7. Train MMD + SCL using training participants only.
    model = train_mmd_scl(
        X_acoustic_train=X_a_train,
        y_acoustic_train=y_a_train,
        X_meg_train=X_m_train,
        y_meg_train=y_m_train,
        fold_number=fold_number
    )

    # 8. Extract 32-D latent representations.
    z_a_train = extract_latent(model, X_a_train)
    z_m_train = extract_latent(model, X_m_train)

    z_a_test = extract_latent(model, X_a_test)
    z_m_test = extract_latent(model, X_m_test)

    X_latent_train = np.vstack([
        z_a_train,
        z_m_train
    ]).astype(np.float32)

    y_latent_train = np.hstack([
        y_a_train,
        y_m_train
    ]).astype(np.int64)

    X_latent_test = np.vstack([
        z_a_test,
        z_m_test
    ]).astype(np.float32)

    y_latent_test = np.hstack([
        y_a_test,
        y_m_test
    ]).astype(np.int64)

    # 9. Train MLP and evaluate held-out participants.
    result = train_and_evaluate_mlp(
        X_train=X_latent_train,
        y_train=y_latent_train,
        X_test=X_latent_test,
        y_test=y_latent_test,
        fold_number=fold_number
    )

    fold_metrics.append({
        key: result[key]
        for key in [
            "accuracy",
            "precision",
            "recall",
            "f1",
            "auc"
        ]
    })

    # 10. Store held-out predictions and embeddings.
    n_a_test = len(test_a_idx)

    pred_a_test = result["predictions"][:n_a_test]
    prob_a_test = result["probabilities"][:n_a_test]

    pred_m_test = result["predictions"][n_a_test:]
    prob_m_test = result["probabilities"][n_a_test:]

    oof_pred_acoustic[test_a_idx] = pred_a_test
    oof_prob_acoustic[test_a_idx] = prob_a_test

    oof_pred_meg[test_m_idx] = pred_m_test
    oof_prob_meg[test_m_idx] = prob_m_test

    fused_acoustic_32D[test_a_idx] = z_a_test
    fused_meg_32D[test_m_idx] = z_m_test

    oof_count_acoustic[test_a_idx] += 1
    oof_count_meg[test_m_idx] += 1

    print("\nHeld-out embedding shapes")
    print("Acoustic:", z_a_test.shape)
    print("MEG     :", z_m_test.shape)


# ============================================================
# OOF ASSIGNMENT CHECK
# ============================================================

print("\n" + "=" * 75)
print("OOF ASSIGNMENT CHECK")
print("=" * 75)

print(
    "Acoustic OOF counts:",
    np.unique(oof_count_acoustic, return_counts=True)
)

print(
    "MEG OOF counts:",
    np.unique(oof_count_meg, return_counts=True)
)

assert np.all(oof_count_acoustic == 1)
assert np.all(oof_count_meg == 1)


# ============================================================
# COMBINE OOF EMBEDDINGS AND PREDICTIONS
# ============================================================

X_fused = np.vstack([
    fused_acoustic_32D,
    fused_meg_32D
]).astype(np.float32)

y_fused = np.hstack([
    y_acoustic,
    y_meg
]).astype(np.int64)

fused_oof_probs = np.hstack([
    oof_prob_acoustic,
    oof_prob_meg
]).astype(np.float32)

fused_oof_preds = np.hstack([
    oof_pred_acoustic,
    oof_pred_meg
]).astype(np.int64)


# ============================================================
# FINAL COMBINED OOF METRICS
# ============================================================

print("\n" + "=" * 75)
print("FINAL CROSS-VALIDATED OOF RESULTS")
print("=" * 75)

print(
    f"Accuracy  : "
    f"{accuracy_score(y_fused, fused_oof_preds):.4f}"
)

print(
    f"Precision : "
    f"{precision_score(y_fused, fused_oof_preds, zero_division=0):.4f}"
)

print(
    f"Recall    : "
    f"{recall_score(y_fused, fused_oof_preds, zero_division=0):.4f}"
)

print(
    f"F1-Score  : "
    f"{f1_score(y_fused, fused_oof_preds, zero_division=0):.4f}"
)

print(
    f"AUC       : "
    f"{roc_auc_score(y_fused, fused_oof_probs):.4f}"
)


# ============================================================
# MEAN AND STANDARD DEVIATION ACROSS OUTER FOLDS
# ============================================================

print("\n" + "=" * 75)
print("MEAN OUTER-FOLD METRICS")
print("=" * 75)

for metric in [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "auc"
]:

    values = np.array(
        [m[metric] for m in fold_metrics],
        dtype=float
    )

    print(
        f"{metric.capitalize():10s}: "
        f"{np.nanmean(values):.4f} "
        f"+/- {np.nanstd(values, ddof=1):.4f}"
    )


# ============================================================
# FINAL OOF LATENT SHAPES
# ============================================================

print("\n" + "=" * 75)
print("FINAL OOF LATENT SHAPES")
print("=" * 75)

print("Acoustic 32-D:", fused_acoustic_32D.shape)
print("MEG 32-D     :", fused_meg_32D.shape)
print("Fused 32-D   :", X_fused.shape)

assert fused_acoustic_32D.shape == (
    len(y_acoustic),
    LATENT_DIM
)

assert fused_meg_32D.shape == (
    len(y_meg),
    LATENT_DIM
)

assert X_fused.shape == (
    len(y_acoustic) + len(y_meg),
    LATENT_DIM
)

assert np.isfinite(X_fused).all()


# ============================================================
# SAVE OOF OUTPUT DATAFRAMES
# ============================================================

latent_columns = [
    f"latent_{i+1}" for i in range(LATENT_DIM)
]

oof_acoustic_df = pd.DataFrame(
    fused_acoustic_32D,
    columns=latent_columns
)

oof_acoustic_df["Status"] = y_acoustic
oof_acoustic_df["OOF_Prediction"] = oof_pred_acoustic
oof_acoustic_df["OOF_Probability"] = oof_prob_acoustic

oof_meg_df = pd.DataFrame(
    fused_meg_32D,
    columns=latent_columns
)

oof_meg_df["Status"] = y_meg
oof_meg_df["OOF_Prediction"] = oof_pred_meg
oof_meg_df["OOF_Probability"] = oof_prob_meg

oof_acoustic_df.to_csv(
    "mira_net_acoustic_oof_embeddings.csv",
    index=False
)

oof_meg_df.to_csv(
    "mira_net_meg_oof_embeddings.csv",
    index=False
)

# Save selected feature names for each outer fold.
selected_feature_rows = []

for fold_number, selected in selected_features_by_fold.items():

    for modality in ["acoustic", "meg"]:

        for rank, feature_name in enumerate(
            selected[modality],
            start=1
        ):

            selected_feature_rows.append({
                "Fold": fold_number,
                "Modality": modality,
                "Rank": rank,
                "Feature": feature_name
            })

pd.DataFrame(selected_feature_rows).to_csv(
    "mira_net_rf_shap_selected_features.csv",
    index=False
)


# ============================================================
# COMPLETION SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("MMD + SCL PIPELINE COMPLETE")
print("=" * 75)

print("✓ No participant-ID columns are used or added")
print("✓ Non-finite values replaced before feature selection")
print("✓ Extreme values clipped for numerical stability")
print("✓ Clipping bounds learned from each training fold only")
print("✓ Median imputation learned from each training fold only")
print("✓ RF-SHAP Top-7 selection performed fold-wise")
print("✓ Acoustic and MEG features selected independently")
print("✓ StandardScaler fitted on training data only")
print("✓ MMD aligns acoustic and MEG latent distributions")
print("✓ SCL uses disease labels to structure embeddings")
print("✓ No domain discriminator or gradient reversal")
print("✓ Original 32-D latent space retained")
print("✓ MLP evaluated on held-out participants")
print("✓ One OOF prediction per participant")
print("✓ Combined OOF metrics calculated")
print("✓ Fold-specific selected features recorded")
print("✓ OOF embeddings saved to CSV")

print("\nOutput arrays:")
print("fused_acoustic_32D:", fused_acoustic_32D.shape)
print("fused_meg_32D     :", fused_meg_32D.shape)
print("X_fused           :", X_fused.shape)

print("\nOutput files:")
print("mira_net_acoustic_oof_embeddings.csv")
print("mira_net_meg_oof_embeddings.csv")
print("mira_net_rf_shap_selected_features.csv")

Device: cpu

Dataset shapes
Acoustic: (193, 69)
MEG     : (127, 105)
Acoustic labels: {np.int64(0): np.int64(66), np.int64(1): np.int64(127)}
MEG labels: {np.int64(0): np.int64(65), np.int64(1): np.int64(62)}

LEAKAGE-FREE RF-SHAP + MMD + SCL + MLP

OUTER FOLD 1/5
Acoustic train/test: 154/39
MEG train/test     : 101/26
Acoustic finite after preprocessing: True
MEG finite after preprocessing: True

Acoustic fold 1 RF-SHAP Top-7 features:
01. Degree-voice breaks (mean |SHAP| = 0.020099)
02. MDVP:Shimmer (mean |SHAP| = 0.019161)
03. Delta1 (mean |SHAP| = 0.019100)
04. RPDE (mean |SHAP| = 0.018424)
05. Delta0 (mean |SHAP| = 0.017792)
06. No. of voice breaks (mean |SHAP| = 0.017317)
07. MFCC5 (mean |SHAP| = 0.017233)

MEG fold 1 RF-SHAP Top-7 features:
01. burst_rate_asym (mean |SHAP| = 0.019693)
02. burst_dur_cv_R_std (mean |SHAP| = 0.014527)
03. burst_dur_cv_L_std (mean |SHAP| = 0.013783)
04. ibi_cv_asym (mean |SHAP| = 0.013600)
05. mean_burst_duration_R_std (mean |SHAP| = 0.013082)
06. m

In [5]:
import torch

# Assuming fused_acoustic_32D and fused_meg_32D are numpy arrays of shape (7350, 32)

# Step 1: Convert numpy arrays to torch tensors (float32)
fused_acoustic_tensor = torch.from_numpy(fused_acoustic_32D).float()
fused_meg_tensor = torch.from_numpy(fused_meg_32D).float()

# Step 2: Concatenate row-wise (along dim=0)
fused_output_tensor = torch.cat([fused_acoustic_tensor, fused_meg_tensor], dim=0)  # Shape: (14700, 32)

# Step 3: Print shape and dtype
print(fused_output_tensor.shape)  # torch.Size([14700, 32])
print(fused_output_tensor.dtype)  # torch.float32


torch.Size([320, 32])
torch.float32


In [6]:
import torch
import numpy as np

# ------------------------
# Step 1: Extract targets correctly
# ------------------------

# Acoustic target from df_acoustic_top7 (last column)
y_acoustic = acoustic_dataset.iloc[:, -1].values.astype(np.int32)   # (7350,)

# MEG target from final_meg_df (last column)
y_meg = meg_dataset.iloc[:, -1].values.astype(np.int32)           # (7350,)

# Convert to tensors (reshape to column)
acoustic_target_tensor = torch.tensor(y_acoustic).long().reshape(-1, 1)   # (7350, 1)
meg_target_tensor      = torch.tensor(y_meg).long().reshape(-1, 1)        # (7350, 1)

# ------------------------
# Step 2: Convert 32D latent features to tensors
# ------------------------

fused_acoustic_tensor = torch.from_numpy(fused_acoustic_32D).float()  # (7350, 32)
fused_meg_tensor      = torch.from_numpy(fused_meg_32D).float()       # (7350, 32)

# ------------------------
# Step 3: Attach target columns
# ------------------------

# Concatenate latent features with target
acoustic_full = torch.cat([fused_acoustic_tensor, acoustic_target_tensor.float()], dim=1)  # (7350, 65)
meg_full      = torch.cat([fused_meg_tensor,      meg_target_tensor.float()],      dim=1)  # (7350, 65)

# ------------------------
# Step 4: Concatenate both datasets
# ------------------------

fused_df = torch.cat([acoustic_full, meg_full], dim=0)  # (14700, 65)

# ------------------------
# Step 5: Separate features and labels for PyTorch
# ------------------------

X = fused_df[:, :-1].float()   # Features (32D) -> float32
y = fused_df[:, -1].long()     # Target -> int32

# ------------------------
# Step 6: Print shapes and dtypes
# ------------------------

print(f"Features shape: {X.shape}, dtype: {X.dtype}")  # (14700, 32), float32
print(f"Labels shape: {y.shape}, dtype: {y.dtype}")    # (14700,), int32

Features shape: torch.Size([320, 32]), dtype: torch.float32
Labels shape: torch.Size([320]), dtype: torch.int64


In [7]:
import torch
import pandas as pd

# fused_df shape = (14700, 33)

# Step 6: Convert fused tensor to numpy
fused_df_np = fused_df.numpy()

# Step 7: Create correct column names for 32 latent features + 1 label
feature_names = [f"feat_{i}" for i in range(32)]
feature_names.append("PD_Label_Combined")

# Step 8: Convert to DataFrame
fused_df_named = pd.DataFrame(fused_df_np, columns=feature_names)

# Step 9: Verify
print(fused_df_named.shape)
print(fused_df_named.head())
print(fused_df_named.tail())

(320, 33)
     feat_0    feat_1    feat_2    feat_3    feat_4    feat_5    feat_6  \
0  0.233974  0.238736 -0.105420 -0.259190 -0.075456 -0.196849  0.169212   
1  0.075918  0.205578 -0.097035  0.002620 -0.278312  0.053203 -0.069111   
2  0.177455  0.223076 -0.148940 -0.169270 -0.030418 -0.092130  0.324149   
3 -0.071962 -0.169995  0.383503  0.061772 -0.125348 -0.086412 -0.151482   
4 -0.103663  0.129344  0.125077 -0.304072  0.190824  0.248345  0.109556   

     feat_7    feat_8    feat_9  ...   feat_23   feat_24   feat_25   feat_26  \
0  0.004837 -0.181179 -0.006858  ...  0.087857 -0.296677 -0.170528  0.128054   
1  0.243775 -0.381592 -0.135363  ... -0.207702 -0.181720  0.162504 -0.175875   
2 -0.090922 -0.114430  0.124807  ...  0.191709 -0.252918 -0.140767  0.021661   
3  0.032667  0.200648 -0.124180  ... -0.244775  0.025612  0.186780 -0.061797   
4  0.002540 -0.224794  0.093679  ...  0.254712 -0.049924 -0.016185  0.039210   

    feat_27   feat_28   feat_29   feat_30   feat_31  PD_La

In [8]:
# Make a copy to be safe (optional)
fused_df_named_int = fused_df_named.copy()

# Convert only the last column to int
fused_df_named_int['PD_Label_Combined'] = fused_df_named_int['PD_Label_Combined'].astype(int)

# Verify
print(fused_df_named_int.dtypes)
print(fused_df_named_int.head())

feat_0               float32
feat_1               float32
feat_2               float32
feat_3               float32
feat_4               float32
feat_5               float32
feat_6               float32
feat_7               float32
feat_8               float32
feat_9               float32
feat_10              float32
feat_11              float32
feat_12              float32
feat_13              float32
feat_14              float32
feat_15              float32
feat_16              float32
feat_17              float32
feat_18              float32
feat_19              float32
feat_20              float32
feat_21              float32
feat_22              float32
feat_23              float32
feat_24              float32
feat_25              float32
feat_26              float32
feat_27              float32
feat_28              float32
feat_29              float32
feat_30              float32
feat_31              float32
PD_Label_Combined      int64
dtype: object
     feat_0    feat_1    feat

In [9]:

# ============================================================
# IMPROVED ENSEMBLE CLASSIFICATION WITH 95% CONFIDENCE INTERVALS
# Stratified 10-Fold CV + Independent Test Set
#
# Models:
#   1. XGBoost
#   2. Random Forest
#   3. HistGradientBoosting
#   4. Weighted Soft Voting
#   5. Stacking
#
# Metrics with 95% CI:
# Accuracy, Precision, Sensitivity, Specificity, F1, AUC
# ============================================================

!pip install -q xgboost scikit-learn statsmodels pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.impute import SimpleImputer
from sklearn.base import clone

from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from statsmodels.stats.proportion import proportion_confint
from xgboost import XGBClassifier


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

feature_cols = [f"feat_{i}" for i in range(32)]

X = fused_df_named_int[feature_cols].copy()
y = fused_df_named_int["PD_Label_Combined"].copy()

X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
y = pd.to_numeric(y, errors="coerce")

valid = y.notna()

X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].astype(int).reset_index(drop=True)

if set(y.unique()) != {0, 1}:
    raise ValueError("Labels must be encoded as 0 and 1.")

if X.shape[1] != 32:
    raise ValueError("Expected exactly 32 embedding features.")

if y.value_counts().min() < 10:
    raise ValueError(
        "Each class needs at least 10 samples for stratified "
        "10-fold cross-validation."
    )

print("X shape:", X.shape)
print("\nClass distribution:")
print(y.value_counts().sort_index())


# ============================================================
# STEP 2: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    stratify=y,
    random_state=42
)

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)


# ============================================================
# STEP 3: MODEL FACTORY
# ============================================================

def create_models(y_fit):

    n_negative = int((y_fit == 0).sum())
    n_positive = int((y_fit == 1).sum())

    scale_pos_weight = n_negative / max(n_positive, 1)

    xgb = XGBClassifier(
        n_estimators=2500,
        max_depth=3,
        learning_rate=0.03,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=5.0,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    rf = RandomForestClassifier(
        n_estimators=2500,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

    hgb = HistGradientBoostingClassifier(
        max_iter=2500,
        learning_rate=0.04,
        max_leaf_nodes=15,
        max_depth=5,
        min_samples_leaf=10,
        l2_regularization=2.0,
        class_weight="balanced",
        early_stopping=False,
        random_state=42
    )

    voting = VotingClassifier(
        estimators=[
            ("xgb", clone(xgb)),
            ("rf", clone(rf)),
            ("hgb", clone(hgb))
        ],
        voting="soft",
        weights=[2, 1, 1],
        n_jobs=1
    )

    stacking = StackingClassifier(
        estimators=[
            ("xgb", clone(xgb)),
            ("rf", clone(rf)),
            ("hgb", clone(hgb))
        ],
        final_estimator=LogisticRegression(
            C=0.5,
            max_iter=2000,
            class_weight="balanced"
        ),
        stack_method="predict_proba",
        passthrough=False,
        cv=StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=123
        ),
        n_jobs=1
    )

    return {
        "XGBoost": xgb,
        "Random Forest": rf,
        "HistGradientBoosting": hgb,
        "Voting Fusion": voting,
        "Stacking Fusion": stacking
    }


# ============================================================
# STEP 4: METRICS + 95% CONFIDENCE INTERVALS
# ============================================================

def calculate_metrics_with_ci(
    y_true,
    y_pred,
    y_prob,
    n_boot=2000,
    random_state=42
):

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    n = len(y_true)

    # ---------------- Point estimates ----------------

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true, y_pred, zero_division=0
    )

    sensitivity = recall_score(
        y_true, y_pred, zero_division=0
    )

    specificity = (
        tn / (tn + fp) if (tn + fp) > 0 else np.nan
    )

    f1 = f1_score(
        y_true, y_pred, zero_division=0
    )

    auc = roc_auc_score(y_true, y_prob)

    results = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "F1": f1,
        "AUC": auc,
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    # ---------------- Wilson 95% CIs ----------------

    proportions = {
        "Accuracy": (tp + tn, n),
        "Precision": (tp, tp + fp),
        "Sensitivity": (tp, tp + fn),
        "Specificity": (tn, tn + fp)
    }

    for metric, (count, denominator) in proportions.items():

        if denominator > 0:
            low, high = proportion_confint(
                count=count,
                nobs=denominator,
                alpha=0.05,
                method="wilson"
            )
        else:
            low, high = np.nan, np.nan

        results[f"{metric} CI Low"] = low
        results[f"{metric} CI High"] = high

    # ---------------- Bootstrap CIs for F1 and AUC ----------------
    # Stratified resampling preserves both class counts.
    # Predictions and probabilities remain paired with labels.

    rng = np.random.default_rng(random_state)

    class_indices = [
        np.flatnonzero(y_true == 0),
        np.flatnonzero(y_true == 1)
    ]

    bootstrap_f1 = []
    bootstrap_auc = []

    for _ in range(n_boot):

        sampled_indices = np.concatenate([
            rng.choice(
                indices,
                size=len(indices),
                replace=True
            )
            for indices in class_indices
        ])

        yt = y_true[sampled_indices]
        yp = y_pred[sampled_indices]
        pr = y_prob[sampled_indices]

        bootstrap_f1.append(
            f1_score(yt, yp, zero_division=0)
        )

        bootstrap_auc.append(
            roc_auc_score(yt, pr)
        )

    for metric, values in [
        ("F1", bootstrap_f1),
        ("AUC", bootstrap_auc)
    ]:

        low, high = np.percentile(
            values,
            [2.5, 97.5]
        )

        results[f"{metric} CI Low"] = low
        results[f"{metric} CI High"] = high

    return results


# ============================================================
# STEP 5: PREPROCESSING
# ============================================================

def make_preprocessor():
    return SimpleImputer(strategy="median")


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CROSS-VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

model_names = [
    "XGBoost",
    "Random Forest",
    "HistGradientBoosting",
    "Voting Fusion",
    "Stacking Fusion"
]

cv_records = []
cv_summary_records = []
test_records = []
confusion_matrices = {}

metrics_to_report = [
    "Accuracy",
    "Precision",
    "Sensitivity",
    "Specificity",
    "F1",
    "AUC"
]


for model_name in model_names:

    print("\n" + "=" * 85)
    print("EVALUATING:", model_name)
    print("=" * 85)

    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        # Fit imputation only on this fold's training partition.
        imputer = make_preprocessor()

        X_tr_imp = imputer.fit_transform(X_tr)
        X_val_imp = imputer.transform(X_val)

        models = create_models(y_tr)
        model = models[model_name]

        model.fit(X_tr_imp, y_tr)

        val_pred = model.predict(X_val_imp)
        val_prob = model.predict_proba(X_val_imp)[:, 1]

        metrics = calculate_metrics_with_ci(
            y_val,
            val_pred,
            val_prob,
            n_boot=2000,
            random_state=1000 + fold
        )

        metrics["Model"] = model_name
        metrics["Fold"] = fold

        fold_metrics.append(metrics)
        cv_records.append(metrics.copy())

        # Print all six metrics and their CIs for each fold.
        print(f"\nFold {fold:02d}/10")

        for metric in metrics_to_report:

            print(
                f"{metric:15s}: {metrics[metric]:.4f} "
                f"(95% CI: "
                f"{metrics[metric + ' CI Low']:.4f}-"
                f"{metrics[metric + ' CI High']:.4f})"
            )

    # ========================================================
    # STEP 7: CV MEAN AND STANDARD DEVIATION
    # ========================================================

    fold_df = pd.DataFrame(fold_metrics)

    print(f"\n{model_name} - 10-Fold CV Summary")
    print("-" * 75)

    summary = {"Model": model_name}

    for metric in metrics_to_report:

        mean_val = fold_df[metric].mean()
        std_val = fold_df[metric].std(ddof=1)

        # Average lower and upper fold CI endpoints.
        mean_ci_low = fold_df[
            f"{metric} CI Low"
        ].mean()

        mean_ci_high = fold_df[
            f"{metric} CI High"
        ].mean()

        summary[f"CV {metric} Mean"] = mean_val
        summary[f"CV {metric} SD"] = std_val
        summary[f"Mean Fold {metric} CI Low"] = mean_ci_low
        summary[f"Mean Fold {metric} CI High"] = mean_ci_high

        print(
            f"{metric:15s}: {mean_val:.4f} +/- {std_val:.4f} "
            f"| Mean fold CI: "
            f"{mean_ci_low:.4f}-{mean_ci_high:.4f}"
        )

    for metric in ["Balanced Accuracy", "MCC"]:

        summary[f"CV {metric} Mean"] = fold_df[metric].mean()
        summary[f"CV {metric} SD"] = fold_df[metric].std(ddof=1)

    cv_summary_records.append(summary)

    # ========================================================
    # STEP 8: FINAL MODEL AND INDEPENDENT TEST SET
    # ========================================================

    final_imputer = make_preprocessor()

    X_train_imp = final_imputer.fit_transform(X_train)
    X_test_imp = final_imputer.transform(X_test)

    final_models = create_models(y_train)
    final_model = final_models[model_name]

    final_model.fit(X_train_imp, y_train)

    test_pred = final_model.predict(X_test_imp)
    test_prob = final_model.predict_proba(X_test_imp)[:, 1]

    test_metrics = calculate_metrics_with_ci(
        y_test,
        test_pred,
        test_prob,
        n_boot=5000,
        random_state=2026
    )

    test_metrics["Model"] = model_name
    test_records.append(test_metrics.copy())

    confusion_matrices[model_name] = confusion_matrix(
        y_test,
        test_pred,
        labels=[0, 1]
    )

    print(f"\n{model_name} - Independent Test Set")
    print("-" * 75)

    for metric in metrics_to_report:

        print(
            f"{metric:15s}: {test_metrics[metric]:.4f} "
            f"(95% CI: "
            f"{test_metrics[metric + ' CI Low']:.4f}-"
            f"{test_metrics[metric + ' CI High']:.4f})"
        )

    print(
        f"Balanced Accuracy: "
        f"{test_metrics['Balanced Accuracy']:.4f}"
    )

    print(f"MCC: {test_metrics['MCC']:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrices[model_name])


# ============================================================
# STEP 9: INDIVIDUAL FOLD RESULTS WITH 95% CIs
# ============================================================

fold_results_df = pd.DataFrame(cv_records)

fold_columns = [
    "Model", "Fold",
    "Accuracy", "Accuracy CI Low", "Accuracy CI High",
    "Precision", "Precision CI Low", "Precision CI High",
    "Sensitivity", "Sensitivity CI Low", "Sensitivity CI High",
    "Specificity", "Specificity CI Low", "Specificity CI High",
    "F1", "F1 CI Low", "F1 CI High",
    "AUC", "AUC CI Low", "AUC CI High",
    "Balanced Accuracy", "MCC",
    "TN", "FP", "FN", "TP"
]

fold_results_df = fold_results_df[fold_columns]

print("\n" + "=" * 100)
print("ALL MODELS: INDIVIDUAL FOLD RESULTS WITH 95% CIs")
print("=" * 100)

display(
    fold_results_df.sort_values(
        ["Model", "Fold"]
    ).round(4)
)


# ============================================================
# STEP 10: CROSS-VALIDATION SUMMARY
# ============================================================

cv_summary_df = pd.DataFrame(cv_summary_records)

print("\n" + "=" * 100)
print("CROSS-VALIDATION MEAN, SD AND MEAN FOLD CIs")
print("=" * 100)

display(cv_summary_df.round(4))


# ============================================================
# STEP 11: INDEPENDENT TEST RESULTS WITH 95% CIs
# ============================================================

test_results_df = pd.DataFrame(
    test_records
).set_index("Model")

test_columns = [
    "Accuracy", "Accuracy CI Low", "Accuracy CI High",
    "Precision", "Precision CI Low", "Precision CI High",
    "Sensitivity", "Sensitivity CI Low", "Sensitivity CI High",
    "Specificity", "Specificity CI Low", "Specificity CI High",
    "F1", "F1 CI Low", "F1 CI High",
    "AUC", "AUC CI Low", "AUC CI High",
    "Balanced Accuracy", "MCC",
    "TN", "FP", "FN", "TP"
]

test_results_df = test_results_df[test_columns]

print("\n" + "=" * 100)
print("INDEPENDENT TEST SET COMPARISON WITH 95% CIs")
print("=" * 100)

display(
    test_results_df.sort_values(
        by="F1",
        ascending=False
    ).round(4)
)


# ============================================================
# STEP 12: VOTING ENSEMBLE RESULTS
# ============================================================

print("\n" + "=" * 100)
print("WEIGHTED VOTING ENSEMBLE RESULTS")
print("=" * 100)

print("Voting weights: XGBoost=2, Random Forest=1, HGB=1")

print("\nVoting fold results:")
display(
    fold_results_df[
        fold_results_df["Model"] == "Voting Fusion"
    ].round(4)
)

print("\nVoting cross-validation summary:")
display(
    cv_summary_df[
        cv_summary_df["Model"] == "Voting Fusion"
    ].round(4)
)

print("\nVoting independent test results:")
display(
    test_results_df.loc[["Voting Fusion"]].round(4)
)

X shape: (320, 32)

Class distribution:
PD_Label_Combined
0    131
1    189
Name: count, dtype: int64

Training shape: (288, 32)
Test shape: (32, 32)

EVALUATING: XGBoost

Fold 01/10
Accuracy       : 0.6897 (95% CI: 0.5077-0.8272)
Precision      : 0.6818 (95% CI: 0.4732-0.8364)
Sensitivity    : 0.8824 (95% CI: 0.6566-0.9671)
Specificity    : 0.4167 (95% CI: 0.1933-0.6805)
F1             : 0.7692 (95% CI: 0.6500-0.8649)
AUC            : 0.7794 (95% CI: 0.5784-0.9314)

Fold 02/10
Accuracy       : 0.7241 (95% CI: 0.5428-0.8530)
Precision      : 0.9091 (95% CI: 0.6226-0.9838)
Sensitivity    : 0.5882 (95% CI: 0.3601-0.7839)
Specificity    : 0.9167 (95% CI: 0.6461-0.9851)
F1             : 0.7143 (95% CI: 0.5000-0.8750)
AUC            : 0.7304 (95% CI: 0.5294-0.9020)

Fold 03/10
Accuracy       : 0.7241 (95% CI: 0.5428-0.8530)
Precision      : 0.7368 (95% CI: 0.5121-0.8819)
Sensitivity    : 0.8235 (95% CI: 0.5897-0.9381)
Specificity    : 0.5833 (95% CI: 0.3195-0.8067)
F1             : 0.7778 (

,Model,Fold,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
20,HistGradientBoosting,1,0.7241,0.5428,0.8530,0.7143,0.5004,0.8619,0.8824,0.6566,...,0.8947,0.7843,0.5931,0.9363,0.6912,0.4213,6,6,2,15
21,HistGradientBoosting,2,0.6897,0.5077,0.8272,0.8333,0.5520,0.9530,0.5882,0.3601,...,0.8667,0.6765,0.4755,0.8676,0.7108,0.4216,10,2,7,10
22,HistGradientBoosting,3,0.5862,0.4074,0.7449,0.6316,0.4104,0.8085,0.7059,0.4687,...,0.8000,0.6863,0.4657,0.8578,0.5613,0.1270,5,7,5,12
23,HistGradientBoosting,4,0.6207,0.4400,0.7731,0.6500,0.4329,0.8188,0.7647,0.5274,...,0.8293,0.6863,0.4706,0.8775,0.5907,0.1931,5,7,4,13
24,HistGradientBoosting,5,0.6552,0.4735,0.8006,0.6667,0.4537,0.8281,0.8235,0.5897,...,0.8500,0.6275,0.3922,0.8431,0.6201,0.2647,5,7,3,14
25,HistGradientBoosting,6,0.5172,0.3443,0.6861,0.5789,0.3628,0.7686,0.6471,0.4130,...,0.7568,0.5686,0.3529,0.7745,0.4902,-0.0203,4,8,6,11
26,HistGradientBoosting,7,0.4828,0.3139,0.6557,0.5625,0.3318,0.7690,0.5294,0.3096,...,0.7222,0.4755,0.2696,0.6912,0.4730,-0.0534,5,7,8,9
27,HistGradientBoosting,8,0.6207,0.4400,0.7731,0.6667,0.4375,0.8372,0.7059,0.4687,...,0.8235,0.6275,0.4020,0.8235,0.6029,0.2090,6,6,5,12
28,HistGradientBoosting,9,0.5714,0.3907,0.7349,0.6667,0.4171,0.8482,0.5882,0.3601,...,0.7879,0.6043,0.3797,0.8182,0.5668,0.1309,6,5,7,10
29,HistGradientBoosting,10,0.5714,0.3907,0.7349,0.6471,0.4130,0.8269,0.6471,0.4130,...,0.8000,0.5829,0.3369,0.8021,0.5508,0.1016,5,6,6,11



CROSS-VALIDATION MEAN, SD AND MEAN FOLD CIs


,Model,CV Accuracy Mean,CV Accuracy SD,Mean Fold Accuracy CI Low,Mean Fold Accuracy CI High,CV Precision Mean,CV Precision SD,Mean Fold Precision CI Low,Mean Fold Precision CI High,CV Sensitivity Mean,...,Mean Fold F1 CI Low,Mean Fold F1 CI High,CV AUC Mean,CV AUC SD,Mean Fold AUC CI Low,Mean Fold AUC CI High,CV Balanced Accuracy Mean,CV Balanced Accuracy SD,CV MCC Mean,CV MCC SD
0,XGBoost,0.6315,0.0737,0.4515,0.7807,0.7000,0.0844,0.4591,0.8607,0.6765,...,0.4994,0.8286,0.6499,0.0836,0.4345,0.8407,0.6227,0.0734,0.2533,0.1534
1,Random Forest,0.6356,0.0747,0.4555,0.7838,0.7018,0.0941,0.4643,0.8595,0.6882,...,0.5119,0.8323,0.6907,0.0592,0.4783,0.8713,0.6244,0.0825,0.2529,0.1669
2,HistGradientBoosting,0.6039,0.0742,0.4251,0.7584,0.6618,0.0747,0.4312,0.8320,0.6882,...,0.4965,0.8131,0.6320,0.0835,0.4138,0.8292,0.5858,0.0762,0.1795,0.1601
3,Voting Fusion,0.6212,0.0701,0.4414,0.7725,0.6801,0.0682,0.4450,0.8469,0.6882,...,0.4996,0.8234,0.6632,0.0764,0.4482,0.8473,0.6070,0.0699,0.2218,0.1475
4,Stacking Fusion,0.6248,0.0807,0.4453,0.7749,0.7074,0.0901,0.4579,0.8693,0.6353,...,0.4668,0.8193,0.6752,0.0717,0.4607,0.8589,0.6230,0.0796,0.2502,0.1658



INDEPENDENT TEST SET COMPARISON WITH 95% CIs


...

,Model,Fold,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
30,Voting Fusion,1,0.7241,0.5428,0.8530,0.7143,0.5004,0.8619,0.8824,0.6566,...,0.8947,0.7941,0.5980,0.9364,0.6912,0.4213,6,6,2,15
31,Voting Fusion,2,0.6897,0.5077,0.8272,0.8333,0.5520,0.9530,0.5882,0.3601,...,0.8667,0.7108,0.5049,0.8873,0.7108,0.4216,10,2,7,10
32,Voting Fusion,3,0.6897,0.5077,0.8272,0.7000,0.4810,0.8545,0.8235,0.5897,...,0.8824,0.7402,0.5392,0.9069,0.6618,0.3444,6,6,3,14
33,Voting Fusion,4,0.6207,0.4400,0.7731,0.6667,0.4375,0.8372,0.7059,0.4687,...,0.8333,0.6912,0.4804,0.8676,0.6029,0.2090,6,6,5,12
34,Voting Fusion,5,0.6207,0.4400,0.7731,0.6500,0.4329,0.8188,0.7647,0.5274,...,0.8237,0.6814,0.4559,0.8676,0.5907,0.1931,5,7,4,13
35,Voting Fusion,6,0.5172,0.3443,0.6861,0.5882,0.3601,0.7839,0.5882,0.3601,...,0.7429,0.5882,0.3725,0.7941,0.5025,0.0049,5,7,7,10
36,Voting Fusion,7,0.5517,0.3755,0.7159,0.6250,0.3864,0.8152,0.5882,0.3601,...,0.7647,0.5392,0.3235,0.7451,0.5441,0.0874,6,6,7,10
37,Voting Fusion,8,0.6552,0.4735,0.8006,0.6842,0.4601,0.8464,0.7647,0.5274,...,0.8485,0.5980,0.3627,0.7941,0.6324,0.2743,6,6,4,13
38,Voting Fusion,9,0.6071,0.4241,0.7643,0.7143,0.4535,0.8828,0.5882,0.3601,...,0.8128,0.6364,0.4225,0.8289,0.6123,0.2194,7,4,7,10
39,Voting Fusion,10,0.5357,0.3581,0.7047,0.6250,0.3864,0.8152,0.5882,0.3601,...,0.7647,0.6524,0.4225,0.8451,0.5214,0.0422,5,6,7,10



Voting cross-validation summary:


,Model,CV Accuracy Mean,CV Accuracy SD,Mean Fold Accuracy CI Low,Mean Fold Accuracy CI High,CV Precision Mean,CV Precision SD,Mean Fold Precision CI Low,Mean Fold Precision CI High,CV Sensitivity Mean,...,Mean Fold F1 CI Low,Mean Fold F1 CI High,CV AUC Mean,CV AUC SD,Mean Fold AUC CI Low,Mean Fold AUC CI High,CV Balanced Accuracy Mean,CV Balanced Accuracy SD,CV MCC Mean,CV MCC SD
3,Voting Fusion,0.6212,0.0701,0.4414,0.7725,0.6801,0.0682,0.445,0.8469,0.6882,...,0.4996,0.8234,0.6632,0.0764,0.4482,0.8473,0.607,0.0699,0.2218,0.1475



Voting independent test results:


,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,Sensitivity CI High,Specificity,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
Model,,,,,,,,,,,,,,,,,,,,,
Voting Fusion,0.5312,0.3645,0.6913,0.5909,0.3873,0.7674,0.6842,0.4601,0.8464,0.3077,...,0.7619,0.6356,0.4251,0.8219,0.496,-0.0086,4,9,6,13
